# Exploratory Data Analysis — YouTube Toxic Comments

## Objetivo

Este notebook estudia el dataset de comentarios de YouTube para fundamentar un sistema interno de triaje que ayude a priorizar revisiones humanas. El EDA observa la estructura, calidad, etiquetas, textos y distribucion por video antes de tomar decisiones de preprocesamiento o modelado.

El analisis parte del dataset raw y no modifica el archivo original. `IsToxic` se trata como una etiqueta del dataset y como referencia inicial documentada, no como una probabilidad, certeza ni decision de moderacion.

# 1. Objetivo y alcance

Realizaremos un analisis exploratorio del dataset de comentarios de YouTube para comprender su estructura, calidad y caracteristicas antes del preprocesamiento y el modelado. Estudiaremos las etiquetas, el contenido de los textos y la distribucion de los comentarios entre videos para aportar evidencia al sistema de triaje y mantener las decisiones bajo control humano.

El analisis parte del dataset raw y el EDA no modificara el dataset original.

# 2. Configuracion y carga de datos

## 2.1 Importacion de librerias

Importamos solo las librerias necesarias para observar y visualizar el dataset.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## 2.2 Carga del dataset

Cargamos el dataset raw mediante una ruta relativa al notebook. La lectura es reproducible y no realiza limpieza, conversion de tipos ni modificaciones sobre el archivo original.

In [2]:
DATA_PATH = Path('../data/raw/youtoxic_english_1000 (1).csv')
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
print(f'Archivo cargado: {DATA_PATH}')
print(f'Tipo: {type(df).__name__}')

### Interpretacion

**Hecho observado:** la lectura del CSV se completa y `df` queda disponible.

**Implicacion:** las comprobaciones siguientes se realizaran sobre una carga reproducible del dataset raw.

# 3. Comprension inicial del dataset

Observamos dimensiones, variables, tipos, una muestra pequena y cardinalidades. La pregunta es: ¿con que estructura basica vamos a trabajar?

In [3]:
label_cols = [col for col in df.columns if col.startswith('Is')]
id_cols = ['CommentId', 'VideoId']
text_col = 'Text'

print(f'Dimensiones: {df.shape}')
print('Columnas:', df.columns.tolist())
print('Tipos:\n', df.dtypes.to_string())
print('Muestra de identificadores y etiquetas:')
display(df[id_cols + label_cols].head(3))
print('Cardinalidades:')
display(df.nunique(dropna=False).rename('n_unique').to_frame())

### Interpretacion

**Hechos observados:** el dataset contiene 1.000 filas y 15 columnas. Incluye `CommentId`, `VideoId`, `Text` y 12 columnas cuyo nombre comienza por `Is`.

**Interpretacion:** la estructura separa identificadores, texto y etiquetas. Las etiquetas deben estudiarse antes de decidir como se utilizaran.

**Implicacion:** la cardinalidad de identificadores y videos sera relevante para valorar duplicados, concentracion por video y evaluacion posterior.

# 4. Calidad de los datos

Medimos ausencias explicitas y encubiertas, duplicados, consistencia de identificadores y valores de etiquetas. La pregunta es: ¿hay problemas observables que debamos documentar antes de transformar los datos? No corregiremos ninguno en esta fase.

In [4]:
missing_report = pd.DataFrame({
    'n_null': df.isna().sum(),
    'pct_null': df.isna().mean().mul(100),
})
text_series = df[text_col].astype('string')
string_cols = df.select_dtypes(include=['object', 'string']).columns
hidden_missing = pd.DataFrame(index=string_cols)
hidden_missing['empty_string'] = df[string_cols].astype('string').apply(lambda col: col.str.len().eq(0).sum())
hidden_missing['whitespace_only'] = df[string_cols].astype('string').apply(lambda col: col.str.strip().eq('').sum())
hidden_missing['literal_missing_token'] = df[string_cols].astype('string').apply(lambda col: col.str.strip().str.lower().isin({'null', 'none', 'nan', 'na'}).sum())
duplicate_report = pd.Series({
    'full_duplicate_rows': int(df.duplicated().sum()),
    'duplicate_comment_id_rows': int(df['CommentId'].duplicated(keep=False).sum()),
    'duplicate_text_rows': int(df[text_col].duplicated(keep=False).sum()),
    'duplicate_text_excess_rows': int(df[text_col].duplicated().sum()),
})
label_values = {col: sorted(df[col].dropna().unique().tolist(), key=str) for col in label_cols}
identifier_report = pd.Series({
    'unique_comment_id': int(df['CommentId'].nunique(dropna=False)),
    'unique_video_id': int(df['VideoId'].nunique(dropna=False)),
    'video_id_name_error_token': int(df['VideoId'].eq('#NAME?').sum()),
})
text_quality = pd.Series({
    'leading_or_trailing_space_rows': int(text_series.str.len().ne(text_series.str.strip().str.len()).sum()),
    'min_chars': int(text_series.str.len().min()),
    'max_chars': int(text_series.str.len().max()),
})
print('Ausencias:')
display(missing_report)
print('Ausencias encubiertas por columna de texto o identificador:')
display(hidden_missing)
print('Duplicados:')
display(duplicate_report.to_frame('count'))
print('Valores observados en etiquetas:')
display(pd.Series(label_values, name='values'))
print('Consistencia de identificadores:')
display(identifier_report.to_frame('value'))
print('Calidad basica del texto:')
display(text_quality.to_frame('value'))

### Interpretacion

**Hechos observados:** no se observan valores nulos ni tokens literales de ausencia en las columnas de texto e identificadores, no hay filas completamente duplicadas ni identificadores de comentario repetidos. Hay textos repetidos, diferencias de espacios y un valor literal `#NAME?` en `VideoId` que se mantienen sin corregir. Las etiquetas se leen como booleanas.

**Interpretacion:** la ausencia de nulos no elimina la necesidad de revisar representaciones encubiertas o duplicacion de textos.

**Implicaciones:** cualquier tratamiento de duplicados o espacios debe decidirse antes de la evaluacion y no debe aplicarse sobre el raw de forma destructiva.

# 5. Analisis de `IsToxic`

Calculamos la distribucion descriptiva de la etiqueta documentada como referencia inicial del MVP. La pregunta es: ¿cuanta cobertura tienen sus dos valores y que implica esto para estudiar una futura priorizacion, sin tratar la etiqueta como una puntuacion de riesgo?

In [5]:
toxic_distribution = (
    df['IsToxic'].value_counts(dropna=False)
    .rename_axis('IsToxic')
    .to_frame('count')
)
toxic_distribution['pct'] = toxic_distribution['count'].div(len(df)).mul(100)
display(toxic_distribution)

### Interpretacion

**Hechos observados:** `IsToxic` contiene 462 valores `True` (46,2%) y 538 valores `False` (53,8%).

**Interpretacion:** en esta muestra la etiqueta no presenta una clase positiva minoritaria extrema.

**Implicacion:** la distribucion permite estudiar una señal de priorizacion descriptiva, pero no determina por si sola el rendimiento futuro ni sustituye la decision humana.

# 6. Analisis de etiquetas secundarias

Analizamos las otras columnas `Is...`, su cobertura y su relacion empirica con `IsToxic`. La pregunta es: ¿que informacion adicional contienen y existe una regla observable que conecte las etiquetas?

In [6]:
secondary_cols = [col for col in label_cols if col != 'IsToxic']
secondary_distribution = pd.DataFrame({
    'positive': df[secondary_cols].sum(),
    'negative': (~df[secondary_cols]).sum(),
})
secondary_distribution['positive_pct'] = secondary_distribution['positive'].div(len(df)).mul(100)
display(secondary_distribution)

secondary_or = df[secondary_cols].any(axis=1)
logic_check = pd.Series({
    'rows_equal': int((df['IsToxic'] == secondary_or).sum()),
    'rows_not_equal': int((df['IsToxic'] != secondary_or).sum()),
    'IsToxic_true_without_secondary': int((df['IsToxic'] & ~secondary_or).sum()),
    'IsToxic_false_with_secondary': int((~df['IsToxic'] & secondary_or).sum()),
})
print('Comprobacion logica:')
display(logic_check.to_frame('count'))

positive_secondary_count = df[secondary_cols].sum(axis=1)
print('Numero de secundarias positivas por comentario:')
display(positive_secondary_count.value_counts().sort_index().rename('rows').to_frame())

cooccurrence = df[secondary_cols].astype(int).T.dot(df[secondary_cols].astype(int))
print('Matriz de coocurrencia:')
display(cooccurrence)

### Interpretacion

**Hechos observados:** `IsToxic` coincide en las 1.000 filas con el OR de las 11 etiquetas secundarias. No hay casos de `IsToxic=True` sin secundaria positiva ni de `IsToxic=False` con secundaria positiva. Las secundarias presentan coberturas muy distintas; `IsHomophobic` e `IsRadicalism` no tienen positivos.

**Interpretacion:** en este archivo `IsToxic` esta logicamente determinada por las secundarias, aunque el repositorio no documenta el proceso original de anotacion.

**Implicaciones:** las etiquetas no deben tratarse como objetivos independientes sin considerar esta dependencia. El dataset no contiene una etiqueta especifica `IsViolent`; `IsThreat` no debe equipararse automaticamente con violencia.

# 7. Analisis descriptivo del texto

Medimos longitud en caracteres y palabras, casos extremos y diferencias descriptivas por `IsToxic`, sin aplicar limpieza NLP. La pregunta es: ¿que variacion estructural tienen los comentarios y que debe tenerse en cuenta para la revision y el modelado posterior?

In [7]:
text_features = pd.DataFrame(index=df.index)
text_features['chars'] = df['Text'].str.len()
text_features['words'] = df['Text'].str.split().str.len()
text_features['IsToxic'] = df['IsToxic']
length_summary = text_features[['chars', 'words']].describe(percentiles=[.01, .25, .5, .75, .99]).T
print('Resumen de longitudes:')
display(length_summary)
print('Resumen por IsToxic:')
display(text_features.groupby('IsToxic')[['chars', 'words']].agg(['count', 'mean', 'median', 'min', 'max']))
print('Comentarios mas cortos y mas largos por caracteres:')
display(text_features['chars'].sort_values().iloc[[0, 1, -2, -1]].to_frame())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=text_features, x='chars', hue='IsToxic', bins=30, element='step', ax=axes[0])
axes[0].set_title('Longitud en caracteres por IsToxic')
axes[0].set_xlabel('Caracteres')
sns.boxplot(data=text_features, x='IsToxic', y='words', ax=axes[1])
axes[1].set_title('Longitud en palabras por IsToxic')
axes[1].set_xlabel('IsToxic')
axes[1].set_ylabel('Palabras')
plt.tight_layout()
plt.show()

### Interpretacion

**Hechos observados:** los comentarios presentan longitudes variables, con casos muy cortos y otros considerablemente largos. Las distribuciones por `IsToxic` deben interpretarse como descriptivas.

**Interpretacion:** la longitud puede afectar a la lectura humana y a cualquier representacion posterior, pero una diferencia descriptiva no demuestra causalidad ni capacidad predictiva.

**Implicacion:** antes del modelado habra que decidir como conservar textos extremos sin eliminarlos automaticamente.

# 8. Analisis por `VideoId`

Analizamos la concentracion de comentarios y etiquetas por video. La pregunta es: ¿puede el agrupamiento por video afectar la interpretacion de resultados o una futura estrategia de evaluacion? No hacemos particion ni afirmamos leakage.

In [8]:
video_summary = df.groupby('VideoId').agg(
    comments=('CommentId', 'size'),
    toxic=('IsToxic', 'sum'),
).assign(toxic_pct=lambda data: data['toxic'].div(data['comments']).mul(100))
print(f'Videos unicos: {df["VideoId"].nunique()}')
display(video_summary.sort_values('comments', ascending=False))

label_by_video = df.groupby('VideoId')[label_cols].sum()
print('Cobertura de etiquetas por video:')
display(label_by_video)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
video_summary['comments'].sort_values(ascending=False).plot.bar(ax=axes[0], title='Comentarios por video')
axes[0].set_xlabel('VideoId')
axes[0].set_ylabel('Comentarios')
video_summary['toxic_pct'].sort_values(ascending=False).plot.bar(ax=axes[1], title='Porcentaje IsToxic por video')
axes[1].set_xlabel('VideoId')
axes[1].set_ylabel('Porcentaje')
plt.tight_layout()
plt.show()

### Interpretacion

**Hechos observados:** el dataset contiene 13 valores unicos de `VideoId` y la cantidad de comentarios no tiene por que ser uniforme entre videos. La proporcion de `IsToxic` puede variar por grupo.

**Interpretacion:** comentarios del mismo video pueden compartir contexto, tema o patrones de anotacion.

**Implicacion:** una futura evaluacion debe estudiar si la particion por comentario sobreestima la generalizacion a nuevos videos. Esto identifica un riesgo potencial, no demuestra leakage por si solo.

# 9. Riesgos y limitaciones para la evaluacion del MVP

La evidencia del EDA debe interpretarse dentro del alcance del sistema de triaje. El dataset puede no representar diversidad de idiomas, dialectos, comunidades, temas, ironia o contexto conversacional.

El archivo no dispone de una etiqueta especifica para contenido violento. Por tanto, no permite medir especificamente su presencia ni evaluar el rendimiento para esa categoria. `IsThreat` no debe equipararse automaticamente con violencia.

El tamano, los duplicados de texto, la concentracion por video, la dependencia entre etiquetas y las categorias con pocos positivos pueden afectar precision, recall, F1, Precision@K y Recall@K. Estas son limitaciones para investigar, no resultados de un modelo.

# 10. Principales hallazgos

## 10.1 Estructura
El dataset tiene 1.000 filas, 15 columnas, un texto principal, dos identificadores y 12 etiquetas booleanas.

## 10.2 Calidad
No se observan nulos ni filas completamente duplicadas; existen textos repetidos y diferencias de espacios que no se han corregido.

## 10.3 IsToxic
`IsToxic` presenta 462 positivos y 538 negativos. Es una etiqueta del dataset y una referencia inicial del producto, no una probabilidad ni una decision de moderacion.

## 10.4 Etiquetas secundarias
La relacion observada entre `IsToxic` y el OR de las secundarias es exacta en este archivo. La cobertura secundaria es desigual y dos categorias no tienen positivos.

## 10.5 Texto
La longitud textual es variable y debe conservarse para investigar extremos sin eliminarlos automaticamente.

## 10.6 Videos
Los comentarios se agrupan en 13 videos, lo que puede influir en la evaluacion de generalizacion.

## 10.7 Limitaciones
No hay etiqueta especifica de violencia ni contexto conversacional completo; el dataset no valida por si solo el MVP completo.

# 11. Implicaciones para las siguientes fases

Antes del preprocesamiento y modelado quedan por decidir, basandose en esta evidencia: tratamiento no destructivo de duplicados y espacios; estrategia frente a etiquetas con cobertura insuficiente; forma de abordar el balance; variables admisibles; particion que controle la concentracion por video; prevencion de leakage; metricas; baseline; evaluacion de priorizacion e incertidumbre.

Estas son decisiones pendientes y no se implementan en este notebook.

# 12. Conclusiones

El dataset disponible permite estudiar una tarea de priorizacion de comentarios con texto, videos e indicadores de toxicidad. Su estructura es manejable, pero presenta dependencias entre etiquetas, textos repetidos, cobertura desigual de categorias y agrupamiento por video.

El EDA no demuestra que el dataset represente todos los contextos de moderacion ni que valide el MVP completo. Tampoco permite medir especificamente contenido violento. Antes de modelar deben definirse el tratamiento de duplicados, la estrategia de evaluacion por video, las metricas de priorizacion y la forma de comunicar incertidumbre manteniendo el control humano.